
####Objetivo del notebook

a. Aplicar la técnica de agrupamiento Kmodes para k=3 para etiquetar cada uno de los casos no fatales de acuerdo al cluster que pertenece 0, 1 o 2. Para la obtener la base de datos clusterizada 'nofatales_clusterizados'.

b. Obterner los centroides de cada cluster y guardar en: centroides_kmodes.

Entrada:

Base de datos: nofatales_muerte_var_significativas , cuyo origen es notebook 01_7

Salida:

Base de datos: nofatales_clusterizados



In [0]:
#verificar si kmodes forma parte de las bibliotecas estándar de Databricks
try:
    from kmodes.kmodes import KModes
    print("La librería kmodes está instalada.")
except ImportError:
    print("La librería kmodes NO está instalada.")

In [0]:
%pip install kmodes

In [0]:
%restart_python

In [0]:
#Verificar la instalación

from kmodes.kmodes import KModes

print(KModes)

In [0]:

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import count, lit, col, when, regexp_replace, concat

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt

# K-Modes
from kmodes.kmodes import KModes

# Tiempo de ejecución
import time

# Ignorar advertencias
import warnings
warnings.filterwarnings("ignore")

In [0]:
#Leer la tabla base
df_base = spark.table(
    "ml_proyecto_7405607705157039.default.nofatales_muerte_var_significativas"
)

In [0]:
df_nofatales = df_base.filter(col("grupo")=="nofatales")
df_muerte = df_base.filter(col("grupo")=="muerte")

In [0]:
print("No fatales:", df_nofatales.count())

print("Muerte:", df_muerte.count())

In [0]:
#Variables del estudio

variables_kmodes = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"

]

In [0]:
# Número de registros
filas = df_nofatales.count()

# Número de variables
columnas = len(variables_kmodes)

print(f"Filas: {filas:,}")
print(f"Columnas: {columnas}")

In [0]:
df_kmodes = df_nofatales.select(variables_kmodes)

In [0]:
#Preparar los datos para K-Modes

#Convertir el data frame de Spark a Pandas

X_cat = df_kmodes.toPandas()

In [0]:
X_cat.head()

In [0]:
X_cat.shape

In [0]:

kmodes_final = KModes(
    n_clusters=3,
    init='Huang',
    n_init=10,
    verbose=1,
    random_state=42
)

clusters_k3 = kmodes_final.fit_predict(X_cat)

In [0]:
#Verificar que clusters_k3 quedó creado

In [0]:
clusters_k3[:10]

In [0]:
#verificar tamaño de los clusters
np.unique(
    clusters_k3,
    return_counts=True
)

In [0]:
#Crear DataFrame con cluster asignado

df_clusterizado = df_nofatales.toPandas()

df_clusterizado["cluster"] = clusters_k3

In [0]:
df_clusterizado.head(5)

In [0]:
df_clusterizado["cluster"].value_counts()

In [0]:
df_clusterizado.dtypes

In [0]:
df_clusterizado["cluster"] = df_clusterizado["cluster"].astype("int64")

In [0]:
df_clusterizado.dtypes

In [0]:
#Convertir nuevamente el data frame clusterizado de pandas a Spark
df_clusterizado_spark = spark.createDataFrame(
    df_clusterizado
)

In [0]:
df_clusterizado_spark.printSchema()

In [0]:
#Guardar tabla Delta nofatales_clusterizados

df_clusterizado_spark.write\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.nofatales_clusterizados"
    )

In [0]:
#verificar que quedó guardada
display(
    spark.table(
        "ml_proyecto_7405607705157039.default.nofatales_clusterizados"
    )
)

In [0]:

(
    spark.table(
        "ml_proyecto_7405607705157039.default.nofatales_clusterizados"
    )
    .groupBy("cluster")
    .agg(count("*").alias("cantidad"))
    .orderBy("cluster")
    .show()
)

In [0]:
#Obtener los centroides del KModes

centroides_kmodes = kmodes_final.cluster_centroids_

centroides_kmodes

In [0]:
variables = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"
]

df_centroides = pd.DataFrame(
    kmodes_final.cluster_centroids_,
    columns=variables
)

df_centroides.insert(0, "cluster", range(len(df_centroides)))

df_centroides

In [0]:
df_centroides_spark = spark.createDataFrame(df_centroides)


In [0]:
display(df_centroides_spark)

In [0]:
(
    df_centroides_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.centroides_kmodes"
    )
)